# Users, Groups & Access Control

## The Linux user model

Linux was designed as a multi-user operating system from day one. A single machine can have hundreds of accounts, each with their own files and processes, isolated from each other. That isolation is built on **user identity**.

There are three rough classes of accounts:

**1. `root` (UID 0)** — the superuser. Can do anything: read any file, kill any process, change any configuration. There is exactly one root account on a system. **Never log in as root for everyday work** — it bypasses all the safety rails the OS provides. Use `sudo` to do specific admin tasks when you need them.

**2. System users (UIDs roughly 1-999)** — accounts for daemons and services. `www-data` runs your web server. `postgres` runs your database. `systemd-network` runs networking. These accounts usually have no password and no login shell — they exist purely to give services an identity so the kernel can control what they can do.

**3. Regular users (UIDs 1000 and up)** — human accounts. Your `ubuntu` or `alice` or `ganesh` account. These have a home directory, a login shell, and a password.

The exact split between system and regular UIDs is controlled by `/etc/login.defs` (with `UID_MIN` and `UID_MAX`). On most modern distros, regular users start at UID 1000. Older RHEL/CentOS used to start at 500.

The principle that holds the whole model together: **the kernel checks the running process's UID against the file's permissions** every time you try to do something. Process credentials (UID, GID) + file metadata (owner, group, mode) → access decision. That's it. That's the whole model. Everything else in this notebook is the plumbing around that one check.

Find out your own identity with **`id`**, **`whoami`**, or **`groups`**:

In [ ]:
!id
!whoami
!groups

## `/etc/passwd` — the user database

`/etc/passwd` is a plain text file. One line per user. Seven colon-separated fields. World-readable (anyone can read it; only root can write to it).

```
ubuntu:x:1000:1000:Ubuntu User,,,:/home/ubuntu:/bin/bash
   │   │  │    │       │            │            │
   │   │  │    │       │            │            └── login shell
   │   │  │    │       │            └────────────── home directory
   │   │  │    │       └─────────────────────────── GECOS (description / full name)
   │   │  │    └─────────────────────────────────── primary group ID (GID)
   │   │  └──────────────────────────────────────── user ID (UID)
   │   └─────────────────────────────────────────── password field — always 'x' on modern systems
   └─────────────────────────────────────────────── username
```

Field by field:

1. **Username** — the login name. 1-32 characters, usually lowercase letters/numbers/underscores.
2. **Password placeholder** — always `x` today. The actual hash is in `/etc/shadow` (next section). Historically `/etc/passwd` itself held password hashes, but that was insecure because `/etc/passwd` is world-readable.
3. **UID** — numeric user ID. The number the kernel actually checks. `0` = root.
4. **GID** — primary group ID. Each user has one primary group; supplementary groups are listed in `/etc/group`.
5. **GECOS** — historically a multi-field comment with full name, office, phone. Today usually just the user's name or blank.
6. **Home directory** — where the user lands after login. Usually `/home/<username>` or `/root` for root.
7. **Login shell** — what runs when the user logs in. Usually `/bin/bash`. Set to `/usr/sbin/nologin` or `/bin/false` to forbid login (common for system users).

Anyone can `cat /etc/passwd` to enumerate users on a system — this is intentional. The file is world-readable because many programs need to look up UIDs to names. But password hashes are *not* in this file.

In [ ]:
!head -5 /etc/passwd
!tail -5 /etc/passwd
!grep -v nologin /etc/passwd | grep -v "/bin/false" | head -10

## `/etc/shadow` — where passwords actually live

`/etc/shadow` holds the **hashed passwords**. Unlike `/etc/passwd`, it's **readable only by root** (permissions `640`, owned by `root:shadow`). Hence the name "shadow" — it shadows `/etc/passwd`.

The format, again colon-separated:

```
ubuntu:$y$j9T$....hash....:19712:0:99999:7:::
   │            │            │  │   │   │ │ │
   │            │            │  │   │   │ │ └── reserved
   │            │            │  │   │   │ └──── account expiration date
   │            │            │  │   │   └────── days before expiry to warn user
   │            │            │  │   └────────── max password age (days, "99999" = never)
   │            │            │  └────────────── min password age (days)
   │            │            └─────────────────  date password last changed (days since 1970)
   │            └──────────────────────────────  password hash (with salt + algorithm marker)
   └───────────────────────────────────────────  username
```

The password field can also be:

- **`!`** or **`!!`** — account is **locked** (no password — can't log in with a password, but can still be reached via SSH key, su from root, etc.)
- **`*`** — the account never had a password; used for system accounts.
- **Empty** — no password required. Disable this everywhere; it's a security disaster.

The hash itself starts with a marker indicating the algorithm:
- `$1$` — MD5 (legacy, weak — don't use)
- `$5$` — SHA-256
- `$6$` — SHA-512 (common default on older systems)
- `$y$` — yescrypt (modern default on newer Debian/Ubuntu)

You don't read `/etc/shadow` directly very often. The commands `passwd`, `chage`, `useradd`, and `usermod` manipulate it for you. If you ever need to inspect it: `sudo cat /etc/shadow`.

## `/etc/group` — groups

`/etc/group` lists groups and their members. One line per group. Four colon-separated fields:

```
sudo:x:27:alice,bob,ganesh
  │  │  │       │
  │  │  │       └── comma-separated list of supplementary members
  │  │  └────────── group ID (GID)
  │  └───────────── password placeholder (rarely used)
  └──────────────── group name
```

A user's **primary group** is the GID from `/etc/passwd` field 4. Files they create are by default owned by their primary group. Most regular users have a primary group matching their username (e.g. user `alice` has primary group `alice`) — the "user private group" model used by most modern distros.

**Supplementary groups** are extra groups the user is a member of, listed in `/etc/group`. A user can be in many groups; the kernel checks all of them when deciding access. Common groups you'll see:

| Group | Purpose |
|---|---|
| **`sudo`** (Debian/Ubuntu) or **`wheel`** (RHEL/Fedora) | members can `sudo` |
| **`adm`** | read system logs (`/var/log/`) |
| **`docker`** | run docker without sudo (effectively root — be careful) |
| **`audio`**, **`video`**, **`render`** | access to audio/video devices |
| **`disk`**, **`tape`** | direct device access |
| **`shadow`** | read `/etc/shadow` |

To see groups for a user:

```bash
groups                       # your own groups
groups alice                 # alice's groups
id alice                     # UID, GID, and all groups in one line
```

To look up users/groups in the system database (which can pull from LDAP, NIS, etc., not just files):

```bash
getent passwd alice          # like grep alice /etc/passwd but consults all sources
getent group sudo            # like grep ^sudo: /etc/group
```

Use `getent` in scripts — it works correctly in environments where users come from a network directory.

In [ ]:
!head -10 /etc/group
!grep -E "^(sudo|wheel|adm|docker):" /etc/group 2>/dev/null
!getent passwd $USER
!groups

## User management — `useradd`, `usermod`, `userdel`

These three commands need `sudo`. They're the low-level tools for managing accounts.

**`useradd`** — create a new user:

```bash
sudo useradd -m -s /bin/bash -c "Alice Smith" alice
```

The flags you'll almost always want:

| Flag | Effect |
|---|---|
| **`-m`** | create the home directory (often forgotten — without `-m` you get a broken account) |
| **`-s SHELL`** | set login shell (e.g. `/bin/bash`) |
| **`-c "GECOS"`** | set the comment / full name |
| **`-G group1,group2`** | add to these supplementary groups |
| **`-u UID`** | use a specific UID (otherwise next free above `UID_MIN`) |
| **`-d /path`** | non-default home directory |
| **`-N`** | don't create a primary group of the same name (use `-g existing-group` instead) |

After `useradd`, the account has **no password set** (locked). Set one with:

```bash
sudo passwd alice
```

There's also a higher-level tool, **`adduser`** (Debian/Ubuntu) or interactive `useradd` modes, that prompts for everything in sequence. The friendlier UI is `adduser`; for scripts, use `useradd`.

**`usermod`** — modify an existing user. Same flag set as `useradd`, plus:

```bash
sudo usermod -aG docker alice            # ADD alice to the docker group (-a is critical!)
sudo usermod -G sudo,docker alice        # SET alice's supplementary groups (overwrites!)
sudo usermod -l new_name old_name        # rename
sudo usermod -L alice                    # lock account (sets password to !hash)
sudo usermod -U alice                    # unlock
sudo usermod -s /bin/zsh alice           # change shell
```

**The `-aG` vs `-G` distinction is a frequent gotcha.** Without `-a` (append), `-G` *replaces* the user's supplementary groups. Always use `-aG` when adding someone to a new group — otherwise you'll accidentally remove them from everything else.

**`userdel`** — delete a user:

```bash
sudo userdel alice                       # delete the account, leave home directory
sudo userdel -r alice                    # also remove home directory and mail spool
```

Be careful with `-r` — it removes the user's files. Make sure no processes are still running as the user (`pgrep -u alice` to check) before deleting.

## Passwords and account aging — `passwd`, `chage`

**`passwd`** — change a password. With no arguments, changes your own; with a username, changes that user's (root only).

```bash
passwd                            # change my password (interactive)
sudo passwd alice                 # change alice's password
sudo passwd -l alice              # lock the account (same as usermod -L)
sudo passwd -u alice              # unlock
sudo passwd -e alice              # expire the password — force change at next login
sudo passwd -d alice              # delete the password (no password required to login — dangerous)
```

**`chage`** — change password aging. Controls how often users must change their password, how soon they're warned, when the account expires.

```bash
sudo chage -l alice                       # list alice's aging info
sudo chage -M 90 alice                    # max password age = 90 days
sudo chage -m 7 alice                     # min password age = 7 days (can't change again within a week)
sudo chage -W 14 alice                    # warn 14 days before expiry
sudo chage -E 2026-12-31 alice            # account expires on this date
sudo chage -E -1 alice                    # remove expiration
sudo chage -d 0 alice                     # force password change at next login
```

For LFCS, know `passwd -l/-u/-e` and `chage -l/-M/-W/-E`. Password policies on production systems usually combine `chage` defaults (in `/etc/login.defs`) with PAM policies (in `/etc/pam.d/common-password` on Debian or `/etc/security/pwquality.conf` on RHEL).

## Group management — `groupadd`, `groupmod`, `groupdel`, `gpasswd`

The group equivalents of the user commands.

```bash
sudo groupadd devs                        # create group 'devs'
sudo groupadd -g 1500 devs                # ...with specific GID
sudo groupmod -n developers devs          # rename devs → developers
sudo groupmod -g 1600 developers          # change GID
sudo groupdel developers                  # delete (only if no user has it as primary group)
```

**`gpasswd`** — manage group membership without `usermod`:

```bash
sudo gpasswd -a alice devs                # add alice to devs
sudo gpasswd -d alice devs                # remove alice from devs
sudo gpasswd -A bob devs                  # make bob an admin of group devs
                                          # (can add/remove members without sudo)
sudo gpasswd -M alice,bob,carol devs      # set the entire member list
```

**A practical workflow** for adding someone to a new group:

```bash
sudo gpasswd -a alice docker
# OR equivalently:
sudo usermod -aG docker alice

# Then alice needs to log out and back in for the new group to take effect in her shell.
# (Or use 'newgrp docker' to start a new shell with that group already active.)
```

**Why log out?** Group memberships are loaded at login. An already-running shell doesn't see new groups until it re-reads the user's identity — which happens at login. The `newgrp` command starts a new shell with a specific group active, which is the workaround.

## Switching users — `su` vs `sudo`

Two ways to run a command as a different user.

**`su` (substitute user)** — start a new shell as another user. Prompts for **that user's password**.

```bash
su                              # become root (deprecated — see below)
su -                            # become root with root's login environment
su alice                        # become alice
su - alice                      # become alice with login environment
su -c "command"                 # run one command as root, then return
```

The **`-`** (or `-l`) flag is the important one. Without it, you become the other user but keep your own environment (cwd, env vars). With it, you simulate a fresh login — runs login scripts, changes to that user's home directory.

**`sudo` (substitute user do)** — run a single command as another user (root by default). Prompts for **your own password**.

```bash
sudo command                    # run as root
sudo -u alice command           # run as alice
sudo -i                         # interactive root shell (like su - root, but with sudo's auth)
sudo -s                         # shell as root with your current env
sudo -k                         # forget cached credentials (next sudo will prompt again)
```

**`sudo` is the modern recommended approach** for almost everything:

| | `su -` | `sudo` |
|---|---|---|
| Asks for whose password? | the target user's | your own |
| Shares the root password? | yes — everyone who needs root knows it | no — root password can be locked |
| Per-command logging? | no | yes (`/var/log/auth.log`) |
| Fine-grained allow list? | no | yes (`/etc/sudoers`) |
| Time-limited? | no | yes (default cache: 5-15 min) |

On Ubuntu and many cloud Linux images, **root is locked** (`passwd -l root`) and there's no root password at all. All admin work goes through `sudo`. This is the right pattern: one shared root account is hard to audit; per-user `sudo` policies are easy.

When to use `su -`: rarely, but useful when you genuinely want to become another non-root user for a while (e.g. testing as `www-data`). `sudo -i -u www-data` does the same thing without sharing passwords.

## `sudo` deep dive — `/etc/sudoers` and `visudo`

`sudo`'s behaviour is configured in `/etc/sudoers` (and files dropped into `/etc/sudoers.d/`). The syntax is its own little language.

**Never edit `/etc/sudoers` directly.** Use **`visudo`**, which checks syntax before saving. A broken sudoers can lock you out of administration entirely.

```bash
sudo visudo                              # edit /etc/sudoers
sudo visudo -f /etc/sudoers.d/myrule     # edit a drop-in file (preferred — won't conflict with package updates)
```

Sudoers lines have the form:

```
WHO  WHERE = (AS_WHOM)  WHAT
```

Examples:

```sudoers
# A specific user can run anything as root, with password
alice  ALL=(ALL:ALL) ALL

# Members of the sudo group (Debian/Ubuntu) — most common rule on the default install
%sudo  ALL=(ALL:ALL) ALL

# Members of wheel (RHEL/Fedora)
%wheel ALL=(ALL) ALL

# Allow alice to restart nginx without a password
alice  ALL=(root) NOPASSWD: /bin/systemctl restart nginx

# Allow the 'devs' group to read all logs
%devs  ALL=(root) NOPASSWD: /usr/bin/journalctl

# Allow alice to become any user except root
alice  ALL=(ALL,!root) ALL
```

Pieces explained:

- **WHO** — username, or `%groupname` for a group.
- **WHERE** — which hosts this rule applies to. `ALL` everywhere; only matters for shared sudoers across many machines.
- **(AS_WHOM)** — `(user:group)` — what identity sudo can switch to. `(ALL)` = any user. Often omitted, defaulting to root.
- **WHAT** — full path(s) to allowed commands, comma-separated. `ALL` = any command.
- **NOPASSWD:** before the command — skip the password prompt for these commands.

**Drop-in files in `/etc/sudoers.d/`** are the preferred way to add rules:

```bash
sudo visudo -f /etc/sudoers.d/ci-runners
# Add:  ci ALL=(root) NOPASSWD: /usr/bin/docker, /usr/bin/podman
```

This keeps your customisations separate from the distro's main `sudoers` (which package updates may rewrite).

To inspect what `sudo` will let you do, run:

```bash
sudo -l                                  # list your sudo privileges
```

This is your first debug step when "sudo isn't working" — it shows exactly which commands and conditions apply.

## Special permission bits — SUID, SGID, sticky

You met the standard `rwxrwxrwx` permissions in notebook 03. There are **three more bits** that change behaviour in specific ways. They show up as modifiers to the existing `x` positions in `ls -l`.

### SUID (Set User ID) — bit `4000`

**On an executable file**: when the program runs, it runs **as the file's owner**, not as the user who launched it.

The canonical example is **`/usr/bin/passwd`**. Regular users need to change their own password — which means writing to `/etc/shadow`. But `/etc/shadow` is owned by root and not writable by anyone else. The solution: `passwd` is owned by root and has the SUID bit set, so when *you* run it, the process runs as root and can write to `/etc/shadow`.

In `ls -l`, SUID shows as an **`s`** in the owner-execute position:

```
-rwsr-xr-x 1 root root 68208 ... /usr/bin/passwd
   ↑
   's' instead of 'x' for owner = SUID set
```

Set with: `chmod u+s file` or `chmod 4755 file` (the leading `4` is the SUID bit).

**SUID is powerful and dangerous.** A bug in a SUID-root program is a privilege escalation vulnerability — anyone can use it to gain root. Modern systems try to minimise SUID binaries; find them with:

```bash
find / -perm -4000 -type f 2>/dev/null
```

You'll see a small list of carefully-audited tools: `passwd`, `sudo`, `mount`, `su`, `ping` (historically), `pkexec`.

### SGID (Set Group ID) — bit `2000`

Two distinct uses, depending on what it's set on:

**On an executable file**: similar to SUID, but the process runs with the file's **group** instead of the user's group. Rare in practice.

**On a directory** (much more common): files created inside that directory inherit the directory's group, instead of the creator's primary group. This is the standard pattern for **shared project directories**:

```bash
sudo mkdir /srv/project
sudo chgrp devs /srv/project
sudo chmod 2775 /srv/project        # 2 = SGID
# Now any file created in /srv/project will be owned by group 'devs',
# regardless of who created it — so all team members can read/write each other's files.
```

In `ls -l`, SGID shows as **`s`** in the group-execute position:

```
drwxrwsr-x 2 root devs 4096 ... /srv/project
        ↑
        's' on group = SGID set
```

### Sticky bit — bit `1000`

**On a directory**: only the file's owner (or root) can delete or rename files within. Other users with write access can still *create* files, but they can only delete their own.

The classic example is **`/tmp`** — anyone can create files there, but you can't delete someone else's:

```
drwxrwxrwt 10 root root 4096 ... /tmp
         ↑
         't' = sticky bit set
```

Set with: `chmod +t dir` or `chmod 1755 dir`.

The sticky bit is exactly the right model for **shared writable directories** — `/tmp`, `/var/tmp`, `/dev/shm`. Without it, anyone could `rm /tmp/important-from-another-user`.

### Numeric form summary

When using numeric `chmod`, prefix the three regular digits with one more:

| Prefix | What |
|---|---|
| **`4XXX`** | SUID |
| **`2XXX`** | SGID |
| **`1XXX`** | sticky |
| **`6XXX`** | SUID + SGID (e.g. `6755`) |
| **`3XXX`** | SGID + sticky |

```bash
chmod 4755 /usr/local/bin/myscript    # SUID + rwxr-xr-x
chmod 2775 /srv/project               # SGID + rwxrwxr-x
chmod 1777 /tmp                       # sticky + rwxrwxrwx
```

In [ ]:
!ls -l /usr/bin/passwd /usr/bin/sudo /usr/bin/su 2>/dev/null
!ls -ld /tmp
!find /usr/bin -perm -4000 -type f 2>/dev/null | head -10

## ACLs — when standard owner/group/other isn't enough

The three-class model (user / group / other) covers most cases but can't express "I want alice to have write access AND I want bob to have read access AND I want everyone else to have nothing." For that, Linux supports **POSIX Access Control Lists (ACLs)**.

ACLs let you grant explicit permissions to **named users** and **named groups** beyond the file's owner and group. They're an extension layered on top of the standard model — the standard owner/group/other bits still exist; ACLs add additional entries.

To see ACLs on a file:

```bash
getfacl /path/to/file
```

A file with no ACLs prints the equivalent of standard permissions:

```
# file: example.txt
# owner: alice
# group: devs
user::rw-
group::r--
other::r--
```

To **add an ACL** entry, use **`setfacl`**:

```bash
setfacl -m u:bob:r-- file.txt              # grant bob read access
setfacl -m g:contractors:rw file.txt       # grant the contractors group read+write
setfacl -m u:carol:rwx project/            # grant carol full access to a directory
setfacl -x u:bob file.txt                  # remove bob's ACL
setfacl -b file.txt                        # remove all ACLs (back to plain perms)
```

After `setfacl`, the standard `ls -l` shows a **`+`** at the end of the permissions to indicate "ACL present":

```
-rw-r--r--+ 1 alice devs 0 ... file.txt
          ↑
          + means there are ACL entries — use getfacl to see them
```

For directories, **default ACLs** apply to new files created inside:

```bash
setfacl -d -m u:bob:rwx /srv/project       # bob will have rwx on any new file under /srv/project
```

The `-d` makes it a *default* (template) ACL applied to newly created entries.

**When to use ACLs**: case-by-case grants where the standard model is awkward. **When not**: as the primary access-control mechanism. ACLs are harder to audit, often forgotten in backups (filesystems must be mounted with `acl` option), and can confuse other admins. Prefer groups + standard permissions when you can.

For LFCS, know that ACLs exist, how to read `getfacl` output, and the basic `setfacl -m` syntax.

In [ ]:
!cd /tmp && touch acl-test.txt && getfacl acl-test.txt 2>/dev/null && rm acl-test.txt

## PAM — Pluggable Authentication Modules (preview)

**PAM** is the framework that handles **authentication** on modern Linux. When you log in, `su`, `sudo`, or any service that needs to verify "is this user really who they claim to be?", it doesn't implement the check itself — it calls PAM.

The benefit: changing how authentication works is a configuration change, not a code change. Want two-factor auth? Drop in a PAM module. Want to authenticate against LDAP or Active Directory? PAM module. Want to enforce password complexity? PAM module.

Configuration lives in **`/etc/pam.d/`** — one file per service:

```
/etc/pam.d/
├── login            # console login
├── sshd             # SSH login
├── sudo             # sudo authentication
├── su               # su authentication
├── passwd           # password change
├── common-auth      # shared auth rules (Debian/Ubuntu)
├── common-account   # shared account rules
├── common-session   # shared session rules
└── ...
```

Each file is a small set of rules, one per line:

```
auth     required   pam_unix.so
auth     required   pam_google_authenticator.so  # 2FA, for example
account  required   pam_unix.so
password required   pam_unix.so   sha512  shadow
session  required   pam_unix.so
```

The four **management groups** PAM uses:

| Type | When it runs |
|---|---|
| **`auth`** | identity verification — "prove you are who you say" |
| **`account`** | account status — "is this account allowed to log in right now?" (expired? locked? time-of-day restricted?) |
| **`password`** | password change — complexity rules, history checks |
| **`session`** | session setup/teardown — mount home directory, log the login, set ulimits |

You'll rarely write PAM config from scratch in LFCS-level work. But you may need to:

- **Read** `/etc/pam.d/sshd` to understand why a login is being denied.
- **Tweak** `pam_pwquality` settings in `/etc/security/pwquality.conf` to enforce stronger passwords.
- **Add 2FA** by installing `pam_google_authenticator` and adding a line to the relevant config.

**A critical warning**: PAM misconfiguration can lock you out. Always keep a root session open while editing PAM files, and have a recovery mechanism (single-user mode, console access) ready.

For LFCS, know: PAM exists, config lives in `/etc/pam.d/`, the four management groups, and that you don't usually write rules from scratch.

## Shell startup files — which one runs when

When you log in or open a new shell, bash reads one or more **startup files** that set up your environment — `$PATH`, aliases, custom prompts, etc. The order they're read depends on **what kind of shell** you're starting.

There are four important files:

| File | Read by |
|---|---|
| **`/etc/profile`** | login shells, every user (system-wide) |
| **`~/.bash_profile`** or **`~/.profile`** | login shells, your account |
| **`/etc/bash.bashrc`** | interactive non-login shells, every user (Debian/Ubuntu has this; RHEL puts the system-wide bashrc in `/etc/bashrc`) |
| **`~/.bashrc`** | interactive non-login shells, your account |

The decision tree:

- **Login shell** — created when you log in to a TTY, ssh in, or run `su - alice` (note the `-`). Reads: `/etc/profile` → `~/.bash_profile` (or `~/.bash_login` or `~/.profile` — the first found).
- **Interactive non-login shell** — opening a new terminal window in a GUI desktop, or running `bash` inside an existing shell. Reads: `/etc/bash.bashrc` → `~/.bashrc`.
- **Non-interactive shell** — running a script. Reads neither, but inherits the environment from its parent.

**The mess**: a common idiom is to have `~/.bash_profile` simply source `~/.bashrc`:

```bash
# ~/.bash_profile
[ -f ~/.bashrc ] && source ~/.bashrc
```

This way, your `~/.bashrc` runs in *both* login and non-login interactive shells — much simpler. Many distros set this up by default.

**What goes where (the convention):**

- **`~/.bashrc`** — aliases, shell functions, prompt customisation (`PS1`), interactive-only settings.
- **`~/.bash_profile` / `~/.profile`** — `export`s of environment variables, `PATH` modifications, things that need to happen once per login.
- **`/etc/profile`** and **`/etc/profile.d/*.sh`** — system-wide login setup, packages drop their setup scripts here.
- **`/etc/bash.bashrc`** — system-wide interactive defaults.

A quick example `~/.bashrc` addition:

```bash
# ~/.bashrc
export EDITOR=vim
alias ll='ls -lah'
alias gs='git status'
PS1='\u@\h \w \$ '
```

After editing, either `source ~/.bashrc` to apply immediately, or open a new shell.

This was alluded to in notebook 03 ("hidden files"). Now you know why those dotfiles matter — they're the per-user customisation layer for every shell you open.